# Population Density Data Integration
This notebook joins population density data with the integrated bird-environment dataset and saves `final5.csv`.

## 1) Import libraries
Use `pandas` for tabular processing and `numpy` for numeric handling.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

## 2) Define file paths
Set input and output files relative to this notebook's folder.

In [ ]:
base_dir = Path('.')
final4_path = base_dir / 'final4.csv'
density_path = base_dir / 'lka_general_2020.csv'
output_path = base_dir / 'final5.csv'
final4_path, density_path, output_path

In [ ]:
for p in [final4_path, density_path]:
    print(f'{p.name}:', 'FOUND' if p.exists() else 'MISSING')

## 3) Load datasets
Read the integrated base data (`final4.csv`) and the population density grid.

In [ ]:
final_df = pd.read_csv(final4_path)
dens_df = pd.read_csv(density_path)
final_df.shape, dens_df.shape

In [ ]:
final_df.head()

## 4) Keep required density columns and standardize numeric fields
Convert coordinate and density fields to numeric to avoid merge issues from text values.

In [ ]:
dens_df = dens_df[['longitude', 'latitude', 'lka_general_2020']].copy()

final_df['decimalLongitude'] = pd.to_numeric(final_df['decimalLongitude'], errors='coerce')
final_df['decimalLatitude'] = pd.to_numeric(final_df['decimalLatitude'], errors='coerce')
dens_df['longitude'] = pd.to_numeric(dens_df['longitude'], errors='coerce')
dens_df['latitude'] = pd.to_numeric(dens_df['latitude'], errors='coerce')
dens_df['lka_general_2020'] = pd.to_numeric(dens_df['lka_general_2020'], errors='coerce')

final_df[['decimalLongitude', 'decimalLatitude']].isna().sum()

## 5) Exact coordinate merge
First attempt a direct longitude-latitude match.

In [ ]:
merged = final_df.merge(
    dens_df,
    how='left',
    left_on=['decimalLongitude', 'decimalLatitude'],
    right_on=['longitude', 'latitude']
)

missing_mask = (
    merged['lka_general_2020'].isna()
    & merged['decimalLongitude'].notna()
    & merged['decimalLatitude'].notna()
)

print('Rows without exact population match:', int(missing_mask.sum()))

## 6) Fill unmatched rows using nearest neighbor (optional)
If `scipy` is available, assign the nearest density point within 0.02 degrees (~2 km).

In [ ]:
if missing_mask.any():
    try:
        from scipy.spatial import cKDTree

        ref_points = dens_df[['longitude', 'latitude']].to_numpy()
        query_points = merged.loc[missing_mask, ['decimalLongitude', 'decimalLatitude']].to_numpy()

        tree = cKDTree(ref_points)
        dist, idx = tree.query(query_points, k=1)

        max_dist = 0.02
        nearest_vals = np.where(
            dist <= max_dist,
            dens_df['lka_general_2020'].to_numpy()[idx],
            np.nan
        )

        merged.loc[missing_mask, 'lka_general_2020'] = nearest_vals
        print('Nearest-neighbor fill complete.')
    except ImportError:
        print('scipy not installed; skipped nearest-neighbor fill.')
        print('Install with: pip install scipy')
else:
    print('No missing rows to fill.')

## 7) Save final dataset and report coverage
Drop helper columns and write `final5.csv`.

In [ ]:
merged = merged.drop(columns=['longitude', 'latitude'], errors='ignore')
merged.to_csv(output_path, index=False)

assigned = int(merged['lka_general_2020'].notna().sum())
print(f'Saved: {output_path}')
print(f'Density assigned for {assigned} / {len(merged)} rows')

## 8) QA: Inspect rows with missing population density
Create a separate file with rows where `lka_general_2020` is null-like.

In [ ]:
output_nulls = base_dir / 'null_lkagenrel2020_rows.csv'
null_like = {'', 'null', 'na', 'n/a', 'nan', 'none'}

tmp = merged.copy()
tmp['lka_general_2020_str'] = tmp['lka_general_2020'].astype(str).str.strip().str.lower()
null_mask = tmp['lka_general_2020'].isna() | tmp['lka_general_2020_str'].isin(null_like)
rows_with_null = tmp.loc[null_mask].drop(columns=['lka_general_2020_str'])
rows_with_null.to_csv(output_nulls, index=False)

print(f'Total rows with null-like population density: {len(rows_with_null)}')
print(f'Saved matching rows to: {output_nulls}')